# CNN 모델 예측 결과 해석하기

이 노트북은 앞선 **CNN_with_your_dataset** 노트북에서 학습하고 저장한 CNN 모델을 불러온 후, 독립적인 Test 이미지에 대해 다음 작업을 수행합니다.

1. Google Drive 연결
2. 저장된 CNN 모델 불러오기
3. Test 폴더의 이미지와 실제 클래스 확인
4. 기존 노트북과 동일한 방식으로 이미지 전처리
5. 모든 Test 이미지에 대한 예측 결과 출력
6. Confusion matrix 출력
7. Grad-CAM을 이용한 중요 영역 시각화
8. Integrated Gradients를 이용한 중요 픽셀 시각화
9. 결과 그림과 표를 Google Drive에 저장

> **중요:** `PROJECT_DIR`, `MODEL_FILE_NAME`, `CLASS_NAMES`를 앞선 학습 노트북과 동일하게 설정해야 합니다.


In [ ]:
# ============================================================
# 1. Google Drive 연결
# ============================================================

from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# ============================================================
# 1-1. 필요한 라이브러리 준비
# ============================================================

# iPhone에서 촬영한 HEIC/HEIF 이미지 지원
!pip -q install pillow-heif

import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageOps
from pillow_heif import register_heif_opener

import tensorflow as tf

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay
)

# HEIC/HEIF 이미지를 Pillow에서 열 수 있도록 등록
register_heif_opener()

print("라이브러리 준비 완료")
print("TensorFlow version:", tf.__version__)


.

.

.
# ★★★★ (1) 프로젝트 경로, 저장 모델, 클래스 이름 설정


In [ ]:
# ============================================================
# 2. 프로젝트 경로 및 모델 설정
# ============================================================

# 앞선 CNN 학습 노트북에서 사용한 프로젝트 폴더와
# 반드시 동일하게 설정합니다.
PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/Project 3"

# Test 이미지 폴더
TEST_DIR = f"{PROJECT_DIR}/Test"

# 앞선 노트북에서 모델이 저장된 폴더
SAVE_DIR = f"{PROJECT_DIR}/Saved_models"

# 불러올 모델의 파일 이름을 정확하게 입력합니다.
# Google Drive의 Saved_models 폴더에서 파일명을 확인하세요.
MODEL_FILE_NAME = (
    "cnn_trainNone_size96_blocks2_epochs50.keras"
)

# 저장된 모델의 전체 경로
MODEL_PATH = f"{SAVE_DIR}/{MODEL_FILE_NAME}"

# 앞선 학습 노트북과 동일한 클래스 이름을 입력합니다.
# 파일명은 계속 Class 1, Class 2, Class 3 형식을 사용합니다.
CLASS_NAMES = [
    "Class 1",
    "Class 2",
    "Class 3"
]

# None이면 Test 폴더의 모든 이미지를 사용합니다.
TEST_SAMPLES_PER_CLASS = None

# 같은 조건에서 같은 이미지 순서가 만들어지도록 하는 난수값
RANDOM_SEED = 42

print("Project folder:", PROJECT_DIR)
print("Test folder:", TEST_DIR)
print("Model path:", MODEL_PATH)


In [ ]:
# ============================================================
# 3. 저장된 CNN 모델 불러오기
# ============================================================

if not Path(MODEL_PATH).exists():
    raise FileNotFoundError(
        "저장된 모델 파일을 찾을 수 없습니다.\n"
        f"현재 설정된 경로: {MODEL_PATH}\n\n"
        "MODEL_FILE_NAME과 PROJECT_DIR을 다시 확인하세요."
    )

# 모델은 추가 학습이 아니라 예측에만 사용하므로
# compile=False로 간단하게 불러옵니다.
model = tf.keras.models.load_model(
    MODEL_PATH,
    compile=False
)

# 저장된 모델의 입력 이미지 크기를 자동으로 확인합니다.
# 예: (None, 96, 96, 3)
model_input_shape = model.input_shape

if (
    len(model_input_shape) != 4
    or model_input_shape[-1] != 3
):
    raise ValueError(
        "이 노트북은 RGB 이미지 분류 CNN 모델을 대상으로 합니다.\n"
        f"현재 모델의 입력 형태: {model_input_shape}"
    )

IMAGE_HEIGHT = int(model_input_shape[1])
IMAGE_WIDTH = int(model_input_shape[2])

if IMAGE_HEIGHT != IMAGE_WIDTH:
    raise ValueError(
        "기존 전처리 방식은 정사각형 이미지를 사용합니다.\n"
        f"현재 모델 입력 크기: {IMAGE_HEIGHT} x {IMAGE_WIDTH}"
    )

IMAGE_SIZE = IMAGE_HEIGHT
NUM_CLASSES = len(CLASS_NAMES)

# 모델 출력 클래스 수와 입력한 클래스 이름 수가 같은지 확인
model_output_classes = int(model.output_shape[-1])

if model_output_classes != NUM_CLASSES:
    raise ValueError(
        "모델의 출력 클래스 수와 CLASS_NAMES의 개수가 다릅니다.\n"
        f"모델 출력 클래스 수: {model_output_classes}\n"
        f"CLASS_NAMES 개수: {NUM_CLASSES}"
    )


def find_last_conv_layer_name(cnn_model):
    """
    Grad-CAM에 사용할 마지막 Conv2D 레이어를 자동으로 찾습니다.
    """

    for layer in reversed(cnn_model.layers):

        if isinstance(layer, tf.keras.layers.Conv2D):
            return layer.name

    raise ValueError(
        "모델에서 Conv2D 레이어를 찾지 못했습니다. "
        "Grad-CAM은 CNN의 convolution layer가 필요합니다."
    )


LAST_CONV_LAYER_NAME = find_last_conv_layer_name(model)

print("모델 불러오기 완료")
print("Model input size:", IMAGE_SIZE, "x", IMAGE_SIZE)
print("Number of classes:", NUM_CLASSES)
print("Last convolution layer:", LAST_CONV_LAYER_NAME)

# 저장된 CNN 구조 확인
model.summary()


In [ ]:
# ============================================================
# 4. Test 폴더의 클래스별 이미지 수 확인
# ============================================================

# 사용할 이미지 확장자
IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png",
    ".bmp", ".webp",
    ".heic", ".heif"
}


def collect_files_by_class(folder_path):
    """
    파일명의 Class 1, Class 2, Class 3 부분을 읽어서
    클래스별로 이미지 파일을 분류합니다.
    """

    folder = Path(folder_path)

    if not folder.exists():
        raise FileNotFoundError(
            f"폴더를 찾을 수 없습니다: {folder_path}"
        )

    files_by_class = {
        class_index: []
        for class_index in range(NUM_CLASSES)
    }

    unrecognized_files = []

    for file_path in sorted(folder.iterdir()):

        if not file_path.is_file():
            continue

        if file_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue

        # 파일명 시작 부분에서 Class 번호를 찾음
        # 예: Class 2 (15).jpg → class_number = 2
        match = re.match(
            r"^Class\s*(\d+)",
            file_path.stem,
            flags=re.IGNORECASE
        )

        if match is None:
            unrecognized_files.append(file_path.name)
            continue

        class_number = int(match.group(1))
        class_index = class_number - 1

        if class_index not in files_by_class:
            unrecognized_files.append(file_path.name)
            continue

        files_by_class[class_index].append(str(file_path))

    return files_by_class, unrecognized_files


# Test 폴더 검사
test_files_all, test_unrecognized = collect_files_by_class(
    TEST_DIR
)


def print_file_counts(title, files_by_class):
    print(f"\n{title}")
    print("-" * 40)

    total_count = 0

    for class_index, class_name in enumerate(CLASS_NAMES):
        count = len(files_by_class[class_index])
        total_count += count

        print(f"{class_name}: {count}개")

    print("-" * 40)
    print(f"Total: {total_count}개")


print_file_counts(
    "Test folder image counts",
    test_files_all
)

# 파일명 형식이 맞지 않는 이미지가 있으면 출력
if test_unrecognized:
    print("\nTest 폴더에서 클래스가 인식되지 않은 파일:")

    for filename in test_unrecognized:
        print(" -", filename)


In [ ]:
# ============================================================
# 5. Test 이미지 선택 및 전처리
# ============================================================

def select_files_per_class(
    files_by_class,
    samples_per_class,
    random_seed
):
    """
    각 클래스에서 지정한 수만큼 이미지를 선택합니다.

    samples_per_class가 None이면 모든 이미지를 사용합니다.
    """

    selected_files = {}

    for class_index in range(NUM_CLASSES):

        class_files = list(files_by_class[class_index])

        # 클래스별로 항상 동일한 순서가 만들어지도록 설정
        rng = np.random.default_rng(
            random_seed + class_index
        )

        random_order = rng.permutation(len(class_files))

        shuffled_files = [
            class_files[index]
            for index in random_order
        ]

        if samples_per_class is None:
            selected_files[class_index] = shuffled_files

        else:
            if len(shuffled_files) < samples_per_class:
                raise ValueError(
                    f"{CLASS_NAMES[class_index]}에 필요한 이미지가 부족합니다.\n"
                    f"필요한 수: {samples_per_class}\n"
                    f"현재 수: {len(shuffled_files)}"
                )

            selected_files[class_index] = (
                shuffled_files[:samples_per_class]
            )

    return selected_files


def load_and_preprocess_images(
    files_by_class,
    image_size
):
    """
    기존 CNN 학습 노트북과 동일한 방식으로 이미지를 처리합니다.

    1. 사진 회전 정보 보정
    2. RGB 이미지로 변환
    3. 중앙 기준 정사각형 크롭
    4. 지정한 크기로 축소
    5. 픽셀 값을 0~1 범위로 정규화
    """

    image_arrays = []
    labels = []
    file_paths = []

    for class_index in range(NUM_CLASSES):

        for file_path in files_by_class[class_index]:

            try:
                with Image.open(file_path) as image:

                    # 스마트폰 사진의 회전 정보 적용
                    image = ImageOps.exif_transpose(image)

                    # RGB 이미지로 변환
                    image = image.convert("RGB")

                    # 중앙을 기준으로 정사각형 크롭 후 크기 변경
                    image = ImageOps.fit(
                        image,
                        (image_size, image_size),
                        method=Image.Resampling.LANCZOS,
                        centering=(0.5, 0.5)
                    )

                    # NumPy 배열로 변환하고 0~1로 정규화
                    image_array = (
                        np.asarray(image, dtype=np.float32)
                        / 255.0
                    )

                image_arrays.append(image_array)
                labels.append(class_index)
                file_paths.append(file_path)

            except Exception as error:
                print(f"이미지를 읽지 못했습니다: {file_path}")
                print("Error:", error)

    return (
        np.asarray(image_arrays, dtype=np.float32),
        np.asarray(labels, dtype=np.int32),
        np.asarray(file_paths)
    )


# Test 이미지는 기본적으로 모두 사용
selected_test_files = select_files_per_class(
    test_files_all,
    TEST_SAMPLES_PER_CLASS,
    RANDOM_SEED
)

# 이미지 전처리 및 CNN 입력 배열 변환
X_test, y_test, test_file_paths = (
    load_and_preprocess_images(
        selected_test_files,
        IMAGE_SIZE
    )
)

if len(X_test) == 0:
    raise ValueError(
        "전처리할 수 있는 Test 이미지가 없습니다."
    )

print("이미지 전처리 완료")
print("Test image array shape:", X_test.shape)
print(
    "\n데이터 배열 형태: "
    "(이미지 개수, 이미지 높이, 이미지 너비, RGB 채널 수)"
)

print("\n전처리 후 클래스별 Test 이미지 수")
print("-" * 40)

for class_index, class_name in enumerate(CLASS_NAMES):
    count = int(np.sum(y_test == class_index))
    print(f"{class_name}: {count}개")


.

.

.
# ★★★★ (2) 모든 Test 이미지 예측 및 기본 성능 확인


In [ ]:
# ============================================================
# 6. 모든 Test 이미지 예측
# ============================================================

# 각 Test 이미지에 대한 클래스별 예측 확률 계산
prediction_probabilities = model.predict(
    X_test,
    verbose=0
)

# 가장 높은 확률을 가진 클래스를 최종 예측으로 선택
predicted_labels = np.argmax(
    prediction_probabilities,
    axis=1
)

# 최종 예측 클래스의 confidence
prediction_confidences = np.max(
    prediction_probabilities,
    axis=1
)

# 전체 Test accuracy
test_accuracy = np.mean(
    predicted_labels == y_test
)

print(f"Test Accuracy: {test_accuracy:.2%}")


# 기본 예측 결과 표 생성
prediction_results = pd.DataFrame({
    "Image Index": np.arange(len(X_test)),

    "File Name": [
        Path(file_path).name
        for file_path in test_file_paths
    ],

    "Actual Class": [
        CLASS_NAMES[label]
        for label in y_test
    ],

    "Predicted Class": [
        CLASS_NAMES[label]
        for label in predicted_labels
    ],

    "Confidence (%)": (
        prediction_confidences * 100
    ).round(2),

    "Correct": (
        y_test == predicted_labels
    )
})

# 각 클래스의 예측 확률을 표에 추가
for class_index, class_name in enumerate(CLASS_NAMES):

    prediction_results[
        f"{class_name} Probability (%)"
    ] = (
        prediction_probabilities[:, class_index]
        * 100
    ).round(2)

pd.set_option("display.max_rows", None)

display(prediction_results)


In [ ]:
# ============================================================
# 7. Confusion Matrix
# ============================================================

confusion_matrix_values = confusion_matrix(
    y_test,
    predicted_labels,
    labels=range(NUM_CLASSES)
)

display_object = ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix_values,
    display_labels=CLASS_NAMES
)

figure, axis = plt.subplots(figsize=(7, 6))

display_object.plot(
    ax=axis,
    values_format="d",
    colorbar=False,
    cmap=plt.cm.Blues
)

plt.title("Confusion Matrix for Test Data")
plt.xticks(rotation=20)
plt.show()


# Grad-CAM과 Integrated Gradients

### Grad-CAM

Grad-CAM은 CNN의 마지막 convolution layer를 사용하여 모델이 분류할 때 **어느 영역을 중요하게 사용했는지** 보여줍니다.

- 넓은 영역 단위의 중요도를 확인하기 좋습니다.
- 마지막 convolution feature map을 사용하므로 결과가 비교적 거칠게 보일 수 있습니다.

### Integrated Gradients

Integrated Gradients는 검은색 기준 이미지에서 실제 입력 이미지까지 픽셀 값을 단계적으로 변화시키며, 각 픽셀이 선택된 클래스의 예측에 얼마나 영향을 주었는지 계산합니다.

- 픽셀 수준의 중요도를 확인할 수 있습니다.
- 이 노트북에서는 양과 음의 기여를 모두 포함한 **중요도 크기**를 시각화합니다.

> 두 방법의 강조 영역은 모델의 판단 과정을 해석하기 위한 단서입니다. 강조된 영역이 실제 원인임을 증명하는 것은 아닙니다.


In [ ]:
# ============================================================
# 8. Grad-CAM 함수 준비
# ============================================================

def normalize_heatmap(heatmap):
    """
    Heatmap 값을 0~1 범위로 변환합니다.
    """

    heatmap = np.asarray(
        heatmap,
        dtype=np.float32
    )

    # 음수 값은 제거
    heatmap = np.maximum(
        heatmap,
        0
    )

    maximum_value = np.max(
        heatmap
    )

    if maximum_value > 0:

        heatmap = (
            heatmap
            / maximum_value
        )

    else:

        heatmap = np.zeros_like(
            heatmap
        )

    return heatmap

# ============================================================
# Grad-CAM 전용 모델 생성
# ============================================================

# 동일한 모델을 이미지마다 다시 만들지 않도록 저장하는 공간
_GRADCAM_MODEL_CACHE = {}


def build_gradcam_model(
    cnn_model,
    last_conv_layer_name
):
    """
    저장 후 불러온 CNN layer들을 새로운 입력에서부터
    순서대로 다시 연결하여 Grad-CAM 전용 모델을 만듭니다.

    반환값:
    1. 마지막 convolution layer의 출력
    2. 최종 분류 결과
    """

    # 저장된 CNN 모델과 동일한 크기의 새로운 입력 생성
    gradcam_input = tf.keras.Input(
        shape=cnn_model.input_shape[1:],
        name="gradcam_input"
    )

    x = gradcam_input
    last_conv_output = None

    # 기존 CNN의 layer와 학습된 weight를 그대로 사용하면서
    # 새로운 입력에서부터 순서대로 다시 연결
    for layer in cnn_model.layers:

        # InputLayer가 목록에 포함되어 있다면 건너뜀
        if isinstance(
            layer,
            tf.keras.layers.InputLayer
        ):
            continue

        x = layer(x)

        # Grad-CAM에 사용할 convolution layer 출력 저장
        if layer.name == last_conv_layer_name:
            last_conv_output = x

    if last_conv_output is None:
        raise ValueError(
            "지정한 convolution layer를 찾지 못했습니다.\n"
            f"현재 설정된 layer: {last_conv_layer_name}"
        )

    # 하나의 새 입력으로부터 convolution 출력과
    # 최종 예측 결과가 연결된 모델 생성
    gradcam_model = tf.keras.Model(
        inputs=gradcam_input,
        outputs=[
            last_conv_output,
            x
        ],
        name="gradcam_model"
    )

    return gradcam_model


def get_gradcam_model(
    cnn_model,
    last_conv_layer_name
):
    """
    동일한 Grad-CAM 모델은 한 번만 생성하고 재사용합니다.
    """

    cache_key = (
        id(cnn_model),
        last_conv_layer_name
    )

    if cache_key not in _GRADCAM_MODEL_CACHE:

        _GRADCAM_MODEL_CACHE[cache_key] = (
            build_gradcam_model(
                cnn_model,
                last_conv_layer_name
            )
        )

    return _GRADCAM_MODEL_CACHE[cache_key]


def make_gradcam_heatmap(
    image_batch,
    cnn_model,
    last_conv_layer_name,
    target_class_index
):
    """
    선택한 클래스에 대한 Grad-CAM heatmap을 계산합니다.

    image_batch 형태:
    (1, 이미지 높이, 이미지 너비, 3)
    """

    # 새 입력에서부터 모든 layer가 다시 연결된
    # Grad-CAM 전용 모델 가져오기
    grad_model = get_gradcam_model(
        cnn_model,
        last_conv_layer_name
    )

    # 입력을 TensorFlow float32 tensor로 변환
    image_batch = tf.convert_to_tensor(
        image_batch,
        dtype=tf.float32
    )

    with tf.GradientTape() as tape:

        # 마지막 convolution 출력과 최종 예측 계산
        convolution_outputs, predictions = grad_model(
            image_batch,
            training=False
        )

        # 선택한 클래스의 예측 점수
        target_score = predictions[
            :,
            target_class_index
        ]

    # 선택한 클래스 점수를 convolution feature map에 대해 미분
    gradients = tape.gradient(
        target_score,
        convolution_outputs
    )

    if gradients is None:
        raise ValueError(
            "Grad-CAM gradient를 계산하지 못했습니다. "
            "Grad-CAM 전용 모델의 연결 상태를 확인해야 합니다."
        )

    # 각 feature map channel의 gradient 평균
    pooled_gradients = tf.reduce_mean(
        gradients,
        axis=(0, 1, 2)
    )

    # Batch 차원 제거
    convolution_outputs = (
        convolution_outputs[0]
    )

    # Feature map과 channel 중요도를 결합
    heatmap = tf.reduce_sum(
        convolution_outputs
        * pooled_gradients,
        axis=-1
    )

    return normalize_heatmap(
        heatmap.numpy()
    )


def resize_heatmap(
    heatmap,
    target_height,
    target_width
):
    """
    작은 Grad-CAM heatmap을 입력 이미지 크기로 확대합니다.
    """

    heatmap_image = Image.fromarray(
        np.uint8(
            heatmap * 255
        )
    )

    heatmap_image = heatmap_image.resize(
        (
            target_width,
            target_height
        ),
        resample=Image.Resampling.BILINEAR
    )

    return (
        np.asarray(
            heatmap_image,
            dtype=np.float32
        )
        / 255.0
    )


def overlay_heatmap(
    original_image,
    heatmap,
    alpha=0.45,
    colormap_name="jet"
):
    """
    원본 이미지 위에 색상 heatmap을 겹쳐 표시합니다.
    """

    heatmap = resize_heatmap(
        heatmap,
        original_image.shape[0],
        original_image.shape[1]
    )

    colormap = plt.get_cmap(
        colormap_name
    )

    colored_heatmap = colormap(
        heatmap
    )[:, :, :3]

    overlay = (
        (1 - alpha)
        * original_image
        + alpha
        * colored_heatmap
    )

    return np.clip(
        overlay,
        0,
        1
    )

In [ ]:
# ============================================================
# 9. Integrated Gradients 함수 준비
# ============================================================

def integrated_gradients(
    image,
    cnn_model,
    target_class_index,
    steps=50,
    batch_size=16,
    baseline=None
):
    """
    Integrated Gradients를 계산합니다.

    image 형태:
    (이미지 높이, 이미지 너비, 3)

    baseline:
    비교의 시작점이 되는 이미지입니다.
    None이면 검은색 이미지를 사용합니다.
    """

    image = tf.convert_to_tensor(
        image,
        dtype=tf.float32
    )

    if baseline is None:
        baseline = tf.zeros_like(
            image
        )
    else:
        baseline = tf.convert_to_tensor(
            baseline,
            dtype=tf.float32
        )

    # 0부터 1까지 단계적으로 변화시키는 alpha 값
    alphas = tf.linspace(
        0.0,
        1.0,
        steps + 1
    )

    gradient_batches = []

    for start_index in range(
        0,
        steps + 1,
        batch_size
    ):

        end_index = min(
            start_index + batch_size,
            steps + 1
        )

        alpha_batch = alphas[
            start_index:end_index
        ]

        # 각 alpha를 이미지의 높이, 너비, 채널에 적용
        alpha_batch = alpha_batch[
            :,
            tf.newaxis,
            tf.newaxis,
            tf.newaxis
        ]

        # 검은색 baseline에서 실제 이미지까지
        # 중간 단계 이미지 생성
        interpolated_images = (
            baseline[tf.newaxis, ...]
            + alpha_batch
            * (
                image[tf.newaxis, ...]
                - baseline[tf.newaxis, ...]
            )
        )

        with tf.GradientTape() as tape:

            tape.watch(
                interpolated_images
            )

            predictions = cnn_model(
                interpolated_images,
                training=False
            )

            target_scores = predictions[
                :,
                target_class_index
            ]

        gradients = tape.gradient(
            target_scores,
            interpolated_images
        )

        gradient_batches.append(
            gradients
        )

    # 모든 중간 단계의 gradient 결합
    total_gradients = tf.concat(
        gradient_batches,
        axis=0
    )

    # Trapezoidal rule을 이용한 적분 근사
    average_gradients = (
        total_gradients[:-1]
        + total_gradients[1:]
    ) / 2.0

    average_gradients = tf.reduce_mean(
        average_gradients,
        axis=0
    )

    # 입력과 baseline의 차이를 반영
    attributions = (
        image - baseline
    ) * average_gradients

    return attributions.numpy()


def make_integrated_gradients_mask(
    attributions,
    percentile=99.0
):
    """
    RGB channel 전체의 attribution 크기를 합하여
    하나의 픽셀 중요도 heatmap을 만듭니다.

    매우 큰 일부 값의 영향을 줄이기 위해
    percentile 기준으로 정규화합니다.
    """

    attribution_mask = np.sum(
        np.abs(attributions),
        axis=-1
    )

    scale_value = np.percentile(
        attribution_mask,
        percentile
    )

    if scale_value > 0:
        attribution_mask = (
            attribution_mask
            / scale_value
        )

    attribution_mask = np.clip(
        attribution_mask,
        0,
        1
    )

    return attribution_mask


.

.

.
# ★★★★ (3) 해석 결과 출력 및 저장 설정


In [ ]:
# ============================================================
# 10. 시각화 및 저장 설정
# ============================================================

# Grad-CAM overlay의 투명도
GRADCAM_ALPHA = 0.45

# Integrated Gradients overlay의 투명도
IG_ALPHA = 0.50

# Integrated Gradients 계산 단계 수
# 값이 클수록 계산은 정밀해지지만 시간이 더 오래 걸립니다.
# 권장값: 25, 50, 100
IG_STEPS = 50

# Integrated Gradients 내부 계산 batch size
IG_BATCH_SIZE = 16

# None이면 모든 Test 이미지를 출력합니다.
# 빠른 확인을 위해 3 또는 5처럼 설정할 수도 있습니다.
MAX_IMAGES_TO_DISPLAY = None

# 결과 그림과 CSV 파일을 Google Drive에 저장할지 설정
SAVE_XAI_RESULTS = True

# 결과 저장 폴더
XAI_SAVE_DIR = (
    f"{PROJECT_DIR}/Explainability_results/"
    f"{Path(MODEL_FILE_NAME).stem}"
)

print("Grad-CAM alpha:", GRADCAM_ALPHA)
print("Integrated Gradients steps:", IG_STEPS)
print("Save results:", SAVE_XAI_RESULTS)

if SAVE_XAI_RESULTS:
    os.makedirs(
        XAI_SAVE_DIR,
        exist_ok=True
    )

    print("Result folder:", XAI_SAVE_DIR)


In [ ]:
# ============================================================
# 11. 모든 Test 이미지의 Grad-CAM 및
#     Integrated Gradients 결과 출력
# ============================================================

if MAX_IMAGES_TO_DISPLAY is None:
    number_of_images = len(X_test)
else:
    number_of_images = min(
        MAX_IMAGES_TO_DISPLAY,
        len(X_test)
    )

print(
    f"총 {number_of_images}개 이미지의 "
    "설명 결과를 생성합니다."
)
print(
    "Integrated Gradients 계산으로 인해 "
    "몇 분 정도 걸릴 수 있습니다.\n"
)


for image_index in range(number_of_images):

    print(
        f"Processing image "
        f"{image_index + 1}/{number_of_images}: "
        f"{Path(test_file_paths[image_index]).name}"
    )

    # 전처리된 입력 이미지
    original_image = X_test[
        image_index
    ]

    actual_class_index = int(
        y_test[image_index]
    )

    predicted_class_index = int(
        predicted_labels[image_index]
    )

    confidence = float(
        prediction_confidences[image_index]
    )

    # --------------------------------------------------------
    # Grad-CAM 계산
    # --------------------------------------------------------

    image_batch = np.expand_dims(
        original_image,
        axis=0
    )

    gradcam_heatmap = make_gradcam_heatmap(
        image_batch=image_batch,
        cnn_model=model,
        last_conv_layer_name=LAST_CONV_LAYER_NAME,
        target_class_index=predicted_class_index
    )

    gradcam_overlay = overlay_heatmap(
        original_image=original_image,
        heatmap=gradcam_heatmap,
        alpha=GRADCAM_ALPHA,
        colormap_name="jet"
    )

    # --------------------------------------------------------
    # Integrated Gradients 계산
    # --------------------------------------------------------

    ig_attributions = integrated_gradients(
        image=original_image,
        cnn_model=model,
        target_class_index=predicted_class_index,
        steps=IG_STEPS,
        batch_size=IG_BATCH_SIZE,
        baseline=None
    )

    ig_heatmap = make_integrated_gradients_mask(
        ig_attributions
    )

    ig_overlay = overlay_heatmap(
        original_image=original_image,
        heatmap=ig_heatmap,
        alpha=IG_ALPHA,
        colormap_name="inferno"
    )

    # --------------------------------------------------------
    # 한 이미지에 대한 전체 결과 시각화
    # --------------------------------------------------------

    figure, axes = plt.subplots(
        1,
        5,
        figsize=(20, 4)
    )

    axes[0].imshow(
        original_image
    )
    axes[0].set_title(
        "Preprocessed Input"
    )

    axes[1].imshow(
        gradcam_heatmap,
        cmap="jet"
    )
    axes[1].set_title(
        "Grad-CAM Heatmap"
    )

    axes[2].imshow(
        gradcam_overlay
    )
    axes[2].set_title(
        "Input + Grad-CAM"
    )

    axes[3].imshow(
        ig_heatmap,
        cmap="inferno"
    )
    axes[3].set_title(
        "Integrated Gradients"
    )

    axes[4].imshow(
        ig_overlay
    )
    axes[4].set_title(
        "Input + IG"
    )

    for axis in axes:
        axis.axis("off")

    correctness_text = (
        "Correct"
        if actual_class_index
        == predicted_class_index
        else "Incorrect"
    )

    figure.suptitle(
        f"File: {Path(test_file_paths[image_index]).name}\n"
        f"Actual: {CLASS_NAMES[actual_class_index]} | "
        f"Predicted: {CLASS_NAMES[predicted_class_index]} | "
        f"Confidence: {confidence:.2%} | "
        f"{correctness_text}",
        fontsize=13
    )

    plt.tight_layout(
        rect=[0, 0, 1, 0.84]
    )

    # 결과 그림을 Google Drive에 저장
    if SAVE_XAI_RESULTS:

        safe_file_stem = re.sub(
            r"[^A-Za-z0-9_-]+",
            "_",
            Path(
                test_file_paths[image_index]
            ).stem
        )

        result_image_path = (
            f"{XAI_SAVE_DIR}/"
            f"{image_index:03d}_"
            f"{safe_file_stem}_explanation.png"
        )

        figure.savefig(
            result_image_path,
            dpi=150,
            bbox_inches="tight"
        )

    plt.show()
    plt.close(figure)

print("\n모든 이미지 처리 완료")


In [ ]:
# ============================================================
# 12. 예측 결과 표를 Google Drive에 저장
# ============================================================

if SAVE_XAI_RESULTS:

    RESULT_TABLE_PATH = (
        f"{XAI_SAVE_DIR}/"
        "test_prediction_results.csv"
    )

    prediction_results.to_csv(
        RESULT_TABLE_PATH,
        index=False
    )

    print("예측 결과 표 저장 완료")
    print("Saved path:")
    print(RESULT_TABLE_PATH)

    print("\n시각화 결과 폴더:")
    print(XAI_SAVE_DIR)

else:
    print(
        "SAVE_XAI_RESULTS가 False이므로 "
        "결과 파일을 저장하지 않았습니다."
    )


# 결과 해석 시 확인할 질문

1. Grad-CAM과 Integrated Gradients가 실제 대상 물체나 결함 영역을 강조했는가?
2. 모델이 대상이 아니라 배경, 그림자, 손, 책상과 같은 요소를 강조하지는 않았는가?
3. 잘못 분류된 이미지에서 강조 영역은 올바르게 분류된 이미지와 어떻게 다른가?
4. Confidence가 높지만 잘못 분류된 사례가 있는가?
5. 데이터 수집 방법을 어떻게 바꾸면 모델이 더 적절한 특징을 학습할 수 있을까?

Grad-CAM과 Integrated Gradients가 서로 다른 영역을 강조할 수도 있습니다. 두 방법은 계산 방식과 해상도가 다르므로, 한 방법만으로 모델의 판단을 단정하지 말고 함께 비교하는 것이 좋습니다.
